# CIFAR-10 EDM: GPU starter for parts 1–3

Derived from the supplied CIFAR-10 EDM lab. The full original is preserved as
cifar10_edm_lab.ipynb. This shorter copy contains only setup, downloads, and
parts 1–3. It adds a CUDA check, automatic PNG export, and the two trajectories
requested by the original part 3 exercise. Write your own observations in
notes.md after examining the results.

Run all cells in this shorter notebook on a DSMLP GPU, from the repository
directory. The first run downloads approximately 720 MB. The figure files
are saved under figures/parts1_3/. This copy does not calculate FID, run the
part 5 evaluation sweep, or start training.

Status: prepared and statically checked; not yet executed on DSMLP.


In [ ]:
# Added starter cell: require a GPU and save each displayed figure.
from pathlib import Path
import matplotlib.pyplot as plt
import torch

assert torch.cuda.is_available(), (
    "CUDA is unavailable. Launch launch-scipy-ml.sh with -g 1 and select "
    "the container's Python kernel before running this notebook."
)
print("GPU:", torch.cuda.get_device_name(0))
FIGURE_DIR = Path("figures/parts1_3")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_NAMES = [
    "01_unconditional_samples.png",
    "02_unconditional_trajectory.png",
    "03_conditional_samples.png",
    "04_cfg_cat_grid.png",
    "05_cat_trajectory_w1.png",
    "06_cat_trajectory_w4.png",
]
_saved_figures = []
# Preserve the original show function even when this setup cell is rerun.
_original_show = getattr(plt.show, "_capstone_original", plt.show)

def _save_and_show(*args, **kwargs):
    for number in plt.get_fignums():
        index = len(_saved_figures)
        if index >= len(FIGURE_NAMES):
            raise RuntimeError("Unexpected extra figure. Restart the kernel and run all cells.")
        destination = FIGURE_DIR / FIGURE_NAMES[index]
        plt.figure(number).savefig(destination, dpi=200, bbox_inches="tight")
        _saved_figures.append(str(destination))
        print("Saved:", destination)
    return _original_show(*args, **kwargs)

_save_and_show._capstone_original = _original_show
plt.show = _save_and_show


In [ ]:
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")   # before torch: Apple GPUs fall back to CPU
import copy, hashlib, io, json, pickle, shutil, sys, tarfile, time, types, urllib.request, zipfile
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

if os.environ.get("LAB_DEVICE"):
    DEVICE = torch.device(os.environ["LAB_DEVICE"])
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
QUICK = DEVICE.type != "cuda"                    # True keeps laptop runs short; set False on a GPU
N_EVAL = int(os.environ.get("LAB_N_EVAL", 200 if QUICK else 1000))   # images per setting in part 5
BATCH = 100 if QUICK else 500                    # images per sampling batch
print(f"device {DEVICE}, QUICK = {QUICK}, {N_EVAL} images per setting")
assert DEVICE.type == "cuda", "Remove a non-CUDA LAB_DEVICE override."


### Downloads
Each file is checked against its SHA-256 checksum, so a broken download is caught, and an interrupted download
resumes when you rerun the cell. CIFAR-10 comes from Hugging Face's copy (`uoft-cs/cifar10`), which downloads
much faster than the original server in Toronto. If you already have the original `cifar-10-python.tar.gz`, or
EDM's files from its README, copy them into `downloads/` and they are used instead. To use your own clone of EDM,
set the environment variable `EDM_DIR` to its folder.

In [ ]:
DL, OUT = Path("downloads"), Path("outputs")
DL.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)
EDM_COMMIT = "008a4e5316c8e3bfe61a62f874bddba254295afb"   # the NVlabs/edm version this notebook was tested with
NV = "https://nvlabs-fi-cdn.nvidia.com/edm"
FILES = {   # file: (address, SHA-256)
    "edm-cifar10-32x32-uncond-vp.pkl": (f"{NV}/pretrained/edm-cifar10-32x32-uncond-vp.pkl",
                                        "4d5dcc1f1d0d41c8934ad21626eeddbdc0460182becf9fc059a0631b1eedb4da"),
    "edm-cifar10-32x32-cond-vp.pkl": (f"{NV}/pretrained/edm-cifar10-32x32-cond-vp.pkl",
                                      "b27385e4f2f8d4d1e2e2c03864c5a9574a0c58ba81eb11625ae6d15c401f4796"),
    "cifar10-32x32.npz": (f"{NV}/fid-refs/cifar10-32x32.npz",
                          "4ada108cdfd8e43409ed427f5f76631beac9b9617cdcaf416d41b1e161d85969"),
    "inception-2015-12-05.pkl": ("https://api.ngc.nvidia.com/v2/models/nvidia/research/stylegan3/versions/1/"
                                 "files/metrics/inception-2015-12-05.pkl",
                                 "a31bd1d4522101109044ec28621a5fcb591ed6115e0d58104fc013fa01ef94f7"),
    "cifar10_resnet56-187c023a.pt": ("https://github.com/chenyaofo/pytorch-cifar-models/releases/download/resnet/"
                                     "cifar10_resnet56-187c023a.pt",
                                     "187c023aee0c9cf3093a682d9447a538cbaf5489f7ae78a7df4edf2246ce380b"),
}
TAR = DL / "cifar-10-python.tar.gz"     # the original CIFAR-10 archive: used if you already have it
if not TAR.exists():                    # otherwise Hugging Face's copy of the same images, a fixed version
    HF = "https://huggingface.co/datasets/uoft-cs/cifar10/resolve/0b2714987fa478483af9968de7c934580d0bb9a2/plain_text"
    FILES["cifar10-train.parquet"] = (f"{HF}/train-00000-of-00001.parquet",
                                      "8428b53a88a11ac374111006708df51469e315a22ac6d66470afd9c78d2ae883")
    FILES["cifar10-test.parquet"] = (f"{HF}/test-00000-of-00001.parquet",
                                     "841389e6f2d64f28bf17310e430aebac20ec3ba611a3c5e231dc93c645ce84de")


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def download(url, path, attempts=8):
    """Download with retries; an interrupted download resumes where it stopped."""
    part = Path(f"{path}.part")
    for _ in range(attempts):
        done = part.stat().st_size if part.exists() else 0
        request = urllib.request.Request(url, headers={"Range": f"bytes={done}-"} if done else {})
        try:
            with urllib.request.urlopen(request, timeout=60) as r, open(part, "ab" if r.status == 206 else "wb") as f:
                shutil.copyfileobj(r, f, 1 << 20)
            break
        except urllib.error.HTTPError as e:
            if e.code == 416:                    # the partial file is already complete
                break
            print(f"  {e}; retrying", flush=True)
        except OSError as e:                     # a timeout or a dropped connection
            print(f"  {e}; retrying", flush=True)
    else:
        raise RuntimeError(f"could not download {url}; run this cell again to resume")
    part.replace(path)


for name, (url, digest) in FILES.items():
    path = DL / name
    if path.exists() and sha256(path) == digest:
        continue
    print("downloading", name, flush=True)
    download(url, path)
    if sha256(path) != digest:
        path.unlink()
        raise RuntimeError(f"{name}: the checksum does not match; run this cell again")

EDM_DIR = Path(os.environ.get("EDM_DIR", DL / f"edm-{EDM_COMMIT}"))
if not EDM_DIR.exists():
    print("downloading EDM's code", flush=True)
    download(f"https://github.com/NVlabs/edm/archive/{EDM_COMMIT}.zip", DL / "edm.zip")
    with zipfile.ZipFile(DL / "edm.zip") as z:
        z.extractall(DL)
sampler_file = EDM_DIR / "torch_utils" / "misc.py"   # EDM's training sampler needs a one-line fix for PyTorch 2.2+
if "super().__init__(dataset)" in sampler_file.read_text():
    if "EDM_DIR" in os.environ:                      # your own clone: we do not edit it for you
        print(f"For train.py with PyTorch 2.2 or later, change super().__init__(dataset) to super().__init__() "
              f"in {sampler_file}")
    else:
        sampler_file.write_text(sampler_file.read_text().replace("super().__init__(dataset)", "super().__init__()"))
try:
    import distutils.util                        # EDM imports one helper from distutils, removed in Python 3.12
except ImportError:
    shim = types.ModuleType("distutils.util")
    shim.strtobool = lambda v: int(str(v).lower() in ("y", "yes", "t", "true", "on", "1"))
    sys.modules["distutils"] = types.ModuleType("distutils")
    sys.modules["distutils"].util = shim
    sys.modules["distutils.util"] = shim
sys.path.insert(0, str(EDM_DIR))
from training.networks import EDMPrecond         # EDM's networks and loss, used in part 6
from training.loss import EDMLoss
print("downloads ready")

### Models, data and plotting
EDM's checkpoints are Python pickles that contain the networks; loading them needs EDM's code (above). We keep the
`ema` entry, the averaged weights that EDM samples with.

In [ ]:
def load_edm(name):
    with open(DL / name, "rb") as f:
        return pickle.load(f)["ema"].to(DEVICE).eval().requires_grad_(False)


uncond = load_edm("edm-cifar10-32x32-uncond-vp.pkl")    # D(x; σ)
cond = load_edm("edm-cifar10-32x32-cond-vp.pkl")        # D(x; σ, c)
CLASSES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]


def load_cifar10(split):
    """CIFAR-10 as uint8 images of shape (n, 3, 32, 32) and integer labels."""
    if TAR.exists():
        names = [f"data_batch_{i}" for i in range(1, 6)] if split == "train" else ["test_batch"]
        images, labels = [], []
        with tarfile.open(TAR) as tar:
            for name in names:
                d = pickle.load(tar.extractfile(f"cifar-10-batches-py/{name}"), encoding="bytes")
                images.append(d[b"data"].reshape(-1, 3, 32, 32))
                labels += d[b"labels"]
        return torch.from_numpy(np.concatenate(images)), torch.tensor(labels)
    import pandas as pd                          # Hugging Face's copy: one PNG per row
    from PIL import Image
    table = pd.read_parquet(DL / f"cifar10-{split}.parquet")
    images = np.stack([np.asarray(Image.open(io.BytesIO(img["bytes"])).convert("RGB")) for img in table["img"]])
    return torch.from_numpy(images).permute(0, 3, 1, 2).contiguous(), torch.tensor(table["label"].to_numpy())


train_images, train_labels = load_cifar10("train")      # 50,000
test_images, test_labels = load_cifar10("test")         # 10,000


def how_long(seconds):
    """A rough duration for progress messages."""
    return f"about {seconds / 60:.0f} min" if seconds >= 90 else f"about {seconds:.0f} s"


def to_uint8(x):
    """Network images in [-1, 1] to uint8 pixels, as EDM's generate.py saves them."""
    return (x * 127.5 + 128).clamp(0, 255).to(torch.uint8).cpu()


def show_grid(rows, title=None):
    """rows: {row label: uint8 images of shape (n, 3, 32, 32)}."""
    ncol = max(len(v) for v in rows.values())
    fig, axes = plt.subplots(len(rows), ncol, figsize=(1.05 * ncol + 1.2, 1.1 * len(rows) + 0.4), squeeze=False)
    for r, (label, images) in enumerate(rows.items()):
        for j, ax in enumerate(axes[r]):
            ax.set_xticks([])
            ax.set_yticks([])
            if j < len(images):
                ax.imshow(images[j].permute(1, 2, 0).numpy())
            else:
                ax.axis("off")
        axes[r, 0].set_ylabel(label, rotation=0, ha="right", va="center", fontsize=9)
    if title:
        fig.suptitle(title)
    fig.tight_layout()
    plt.show()


print(f"{len(train_images)} training and {len(test_images)} test images")

## 1. Unconditional samples

The sampler is EDM's: Heun's method over 18 noise levels from $\sigma = 80$ down to 0, which costs 35 network
evaluations per image. At each level the network estimates the clean image, and the sampler moves $x$ toward it.
It is `edm_sampler` from EDM's `generate.py` with its default settings, written out here in float32 so that it
also runs on Apple GPUs.

The starting noise comes from a seed, so a seed gives the same starting noise in every part, and settings can be
compared on the same noise.

In [ ]:
def edm_sample(denoise, n, seed, steps=18, sigma_min=0.002, sigma_max=80.0, rho=7.0, record=False):
    """EDM's deterministic sampler. denoise(x, sigma) returns the clean-image estimate.
    With record=True it also returns, for each step, the noise level σ, the noisy image x and the estimate D(x; σ)."""
    i = torch.arange(steps, dtype=torch.float64)
    sigmas = (sigma_max ** (1 / rho) + i / (steps - 1) * (sigma_min ** (1 / rho) - sigma_max ** (1 / rho))) ** rho
    sigmas = sigmas.tolist() + [0.0]
    x = torch.randn(n, 3, 32, 32, generator=torch.Generator().manual_seed(seed)).to(DEVICE) * sigmas[0]
    history = []
    for s, s_next in zip(sigmas[:-1], sigmas[1:]):
        estimate = denoise(x, s)
        if record:
            history.append((s, x.cpu(), estimate.cpu()))
        d = (x - estimate) / s                               # direction from the clean estimate to x
        x_next = x + (s_next - s) * d                        # Euler step to the next noise level
        if s_next > 0:                                       # Heun: average with the direction there
            x_next = x + (s_next - s) * (d + (x_next - denoise(x_next, s_next)) / s_next) / 2
        x = x_next
    return (x, history) if record else x


@torch.no_grad()
def D(net, x, sigma, labels=None):
    """The clean-image estimate: D(x; σ) if labels is None, else D(x; σ, c) for the class indices in labels."""
    s = torch.full((len(x),), sigma, device=x.device)
    c = None if labels is None else F.one_hot(labels, 10).float().to(x.device)
    return net(x, s, c).float()


x = edm_sample(lambda x, s: D(uncond, x, s), n=30, seed=0)
show_grid({f"images {10 * k + 1}–{10 * k + 10}": to_uint8(x[10 * k:10 * k + 10]) for k in range(3)},
          title="unconditional model, seed 0")

### Inside the sampler: the trajectory and the denoiser's estimates
At each noise level the sampler holds a noisy image $x$ and asks the network for its estimate of the clean image,
$D(x;\sigma)$. The figure follows three images from $\sigma = 80$ down to $\sigma = 0.002$. For each image, the
upper row is the noisy image and the lower row is $D(x;\sigma)$. The noisy image is shown as the course's
$x_t = x/\sqrt{1+\sigma^2}$, which keeps it in the display range.

**Before running:** what will $D(x;\sigma)$ look like at $\sigma = 80$? (In code session 2, the MNIST denoiser's
guess at high noise was an average digit.) At which noise level do you expect the content of the final image to
be settled?

In [ ]:
def show_trajectory(history, rows=3, steps=(0, 3, 6, 8, 10, 12, 14, 17), title=None):
    """For the first `rows` images: the noisy image (as x_t) above the estimate D(x; σ), at several steps."""
    fig, axes = plt.subplots(2 * rows, len(steps), figsize=(1.25 * len(steps) + 1.4, 2.6 * rows), squeeze=False)
    for j, k in enumerate(steps):
        s, x, estimate = history[k]
        for i in range(rows):
            axes[2 * i, j].imshow(to_uint8(x[i] / (1 + s ** 2) ** 0.5).permute(1, 2, 0).numpy())
            axes[2 * i + 1, j].imshow(to_uint8(estimate[i]).permute(1, 2, 0).numpy())
        axes[0, j].set_title(f"σ = {s:.2g}", fontsize=9)
    for ax in axes.flat:
        ax.set_xticks([])
        ax.set_yticks([])
    for i in range(rows):
        axes[2 * i, 0].set_ylabel("noisy x", rotation=0, ha="right", va="center", fontsize=9)
        axes[2 * i + 1, 0].set_ylabel("D(x; σ)", rotation=0, ha="right", va="center", fontsize=9)
    if title:
        fig.suptitle(title)
    fig.tight_layout()
    plt.show()


_, history = edm_sample(lambda x, s: D(uncond, x, s), n=3, seed=0, record=True)
show_trajectory(history, title="unconditional model: the noisy image and the denoiser's estimate at each noise level")

## 2. Conditional samples

The conditional model takes the class as a one-hot vector. Each row requests one class, and each column starts
from the same noise. **Before running:** what do you expect to stay the same down a column?

In [ ]:
rows = {}
for c in range(10):
    labels = torch.full((8,), c)
    rows[CLASSES[c]] = to_uint8(edm_sample(lambda x, s: D(cond, x, s, labels), n=8, seed=1))
show_grid(rows, title="conditional model: one class per row, the same starting noise in each column")

## 3. CFG with two networks

CFG mixes the two estimates:
$$D_w(x;\sigma,c) = D(x;\sigma) + w\,\big(D(x;\sigma,c) - D(x;\sigma)\big).$$
$w = 0$ gives the unconditional model and $w = 1$ the conditional one; $w > 1$ pushes further along the
difference that the class makes.

On MNIST one network gave both estimates, because it was trained with the label replaced by "no label" (10) 10%
of the time. **EDM's conditional network was trained without that, so it has no "no label" input.** Its code has
the switch, `label_dropout` (printed below), but EDM's `train.py` never turns it on, and the checkpoint records 0.
So here the unconditional estimate comes from the separate unconditional network: CFG with two networks. Both
were trained on the same images with the same architecture and noise levels, so their estimates can be mixed.

In [ ]:
source = (EDM_DIR / "training" / "networks.py").read_text().splitlines()
k = next(i for i, line in enumerate(source) if "self.training and self.label_dropout" in line)
print("EDM, training/networks.py, inside SongUNet.forward:")
print("\n".join(source[k - 2:k + 3]))
print("\nlabel_dropout stored in the released conditional network:", cond.model.label_dropout)

**Before running:** what should change as $w$ goes from 1 to 4?

In [ ]:
def D_cfg(x, sigma, labels, w):
    """CFG with two networks: D(x; σ) + w (D(x; σ, c) − D(x; σ))."""
    if w == 1:
        return D(cond, x, sigma, labels)                     # one network evaluation instead of two
    d_uncond = D(uncond, x, sigma)
    if w == 0:
        return d_uncond
    return d_uncond + w * (D(cond, x, sigma, labels) - d_uncond)


W = [0, 1, 1.5, 2, 3, 4]
labels = torch.full((8,), CLASSES.index("cat"))
show_grid({f"w = {w:g}": to_uint8(edm_sample(lambda x, s: D_cfg(x, s, labels, w), n=8, seed=2)) for w in W},
          title="CFG toward cat: the same starting noise in each column")

**Your turn: trajectories with a condition.** Make the trajectory figure from part 1 for the conditional model
($w = 1$) and for CFG at $w = 4$, toward one class and from the same seed. `edm_sample(..., record=True)` returns
the images and the history; `show_trajectory(history)` draws it. Then answer in your notes:
1. At which noise level does the class first show in $D(x;\sigma)$? Is it earlier with $w = 4$?
2. From which noise level on does the image barely change?
3. How do the $w = 4$ estimates differ from the $w = 1$ ones at low noise?

In [ ]:
# Added implementation of the original part 3 trajectory exercise.
# Three samples, class cat, seed 0 for BOTH guidance weights.
trajectory_labels = torch.full((3,), CLASSES.index("cat"))
for guidance_weight in (1, 4):
    _, trajectory_history = edm_sample(
        lambda x, sigma: D_cfg(x, sigma, trajectory_labels, guidance_weight),
        n=3,
        seed=0,
        record=True,
    )
    show_trajectory(
        trajectory_history,
        title=f"cat: w = {guidance_weight}, seed 0",
    )


In [ ]:
# Added starter cell: record the environment only after all six figures succeed.
import importlib.metadata
import json
import platform
from datetime import datetime, timezone

assert len(_saved_figures) == 6, f"Expected 6 figures, got {len(_saved_figures)}"
runtime = {
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "pytorch_cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "edm_commit": EDM_COMMIT,
    "packages": {
        name: importlib.metadata.version(name)
        for name in ("numpy", "scipy", "matplotlib", "pandas", "pyarrow", "Pillow")
    },
    "figure_paths": _saved_figures,
    "sampler": {"steps": 18, "sigma_min": 0.002, "sigma_max": 80.0, "rho": 7.0},
    "trajectory_class": "cat",
    "trajectory_seed": 0,
    "trajectory_guidance_weights": [1, 4],
}
(FIGURE_DIR / "run_environment.json").write_text(json.dumps(runtime, indent=2) + "\n")
print("Finished parts 1–3. Inspect the six figures, then fill in notes.md.")
